# ANN Orchestrator For Multi-Run Ensembling and Checkpointing

This notebook defines a reusable orchestrator that:
1. Configures a set of runs (architectures, hyperparameters, number of seeds).
2. Runs expanding-window forecasts and top-k ensembling (same logic as `compute_top_k_ensemble`).
3. Saves model checkpoints at each refit step (DeepSHAP-friendly: torch `state_dict` + preprocessing state).
4. Saves performance tuples per maturity: `(maturity, r2_oos, rsz_pval)`.

The example at the bottom runs a simple forward-only ANN with `n_models=10` and `k_top=1`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import seaborn as sns
import utils.base_utils as bu
import utils.window_utils as wu
import numpy as np
import matplotlib.pyplot as plt
from utils.publication_lags import apply_fred_md_publication_lag
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array
import torch

repo_root = os.path.abspath('../..')

start_date = '1971-08-31'
# end_date = '2025-06-30' # 2018-12-31 for Bianchi period
end_date = '2018-12-31' 
MATURITIES = ['24', '36', '48', '60', '84', '120']
YIELDS = 'lw' # Alternatively, 'lw' for liu-wu yields
 
# end_date = '2025-06-30' # kr and gsw end date
maturities = [str(i) for i in range(12, 121) if i % 12 == 0] # select only yearly maturities

yields = bu.get_yields(type=YIELDS, start=start_date, end=end_date, maturities=maturities) # type can be kr, lw, gsw
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna() # horizon=12 means holding for 12 months

# adjust fred_md start_date by 6 months to fetch enough data for shifting
fred_md_start_date = pd.to_datetime(start_date) - pd.DateOffset(months=6)
fred_md_raw = bu.get_fred_data('data/2026-01-MD.csv', start=fred_md_start_date, end=end_date) # this is aligned to the last day of the previous month, so we get the same number of observations as the yields data

monthly_yields = bu.get_yields(type=YIELDS, start=start_date, end=end_date, maturities=[str(i) for i in range(1, 121)]) # needed for monthly holding period excess returns. Not available for gsw
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna() # calculate monthly excess returns for robustness

# If wanted, apply per-series publication lag to latest-snapshot macro data
# from utils.publication_lags import apply_fred_md_publication_lag
# fred_md = apply_fred_md_publication_lag(fred_md_raw)  
# For results in paper, we naively shift all FRED-MD series by 1 month
# to reflect publication lag:
fred_md = fred_md_raw.shift(1)

fred_md_realtime = pd.read_csv(
    os.path.join(repo_root, 'data', 'ALFRED', 'simple_outputs', 'realtime_final_balanced.csv'),
    parse_dates=True,
    index_col=0
 )
fred_md_realtime.index.name = 'date'

# Drop TWEXAFEGSMTHx and ACOGNO as they start late
fred_md = fred_md.drop(columns=['TWEXAFEGSMTHx', 'ACOGNO'])
fred_md_realtime = fred_md_realtime.drop(columns=['TWEXAFEGSMTHx', 'ACOGNO'], errors='ignore')
# Finally, revert fred_md to start_date, after transformations and lag adjustments
fred_md = fred_md[start_date:end_date]
fred_md_realtime = fred_md_realtime[start_date:end_date]

# Drop dates outside the xr range
yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]
fred_md_realtime = fred_md_realtime.loc[fred_md_realtime.index <= xr.index[-1]].reindex(xr.index)
monthly_xr = monthly_xr.loc[monthly_xr.index <= xr.index[-1]]

# Construct X with 3-level MultiIndex: (source, group, series)
s2g = build_full_group_mapping(fred_md, forward, yields)

X = pd.concat([fred_md, forward, yields],
               axis=1,
               keys=['fred', 'forward', 'yields'])

X = add_group_level(X, s2g, level_name='group')
X = X.sort_index(axis=1, level='group')
groups = groups_as_array(X, level='group')

X_realtime = pd.concat([fred_md_realtime, forward, yields],
                        axis=1,
                        keys=['fred', 'forward', 'yields'])
X_realtime = add_group_level(X_realtime, s2g, level_name='group')
X_realtime = X_realtime.sort_index(axis=1, level='group')
groups_realtime = groups_as_array(X_realtime, level='group')

y_all = xr[MATURITIES].values
dates = xr.index

/home/ulrikts/Documents/NTNU/TIO4900-Replication/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:219: UserWarning: The following series are defined in get_fredmd_grouping() but are not present in the FRED-MD data: ['ACOGNO', 'TWEXAFEGSMTHx']. They may have been dropped or renamed.
  warnings.warn(
/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:168: UserWarning: 2 entries in series_to_group are not present in the DataFrame columns: ['ACOGNO', 'TWEXAFEGSMTHx']
  warnings.warn(
/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:168: UserWarning: 2 entries in series_to_group are not present in the DataFrame columns: ['ACOGNO', 'TWEXAFEGSMTHx']
  warnings.warn(


In [2]:
def compute_top_k_ensemble(forecasts_array: np.ndarray, val_losses_array: np.ndarray, k: int):
    # Same ensembling logic as existing notebook code: top-k per maturity and date by val loss.
    T, n_seeds, n_outputs = forecasts_array.shape
    ensemble_forecast = np.full((T, n_outputs), np.nan)
    topk_indices = np.full((T, n_outputs, min(k, n_seeds)), -1, dtype=int)

    for t in range(T):
        for m in range(n_outputs):
            v_losses = val_losses_array[t, :, m]
            valid_idx = np.where(~np.isnan(v_losses))[0]
            if len(valid_idx) == 0:
                continue

            actual_k = min(k, len(valid_idx))
            sorted_valid_idx = valid_idx[np.argsort(v_losses[valid_idx])]
            selected = sorted_valid_idx[:actual_k]

            topk_indices[t, m, :actual_k] = selected
            ensemble_forecast[t, m] = np.mean(forecasts_array[t, selected, m], axis=0)

    return ensemble_forecast, topk_indices


def _extract_scaler_state(scaler):
    if scaler is None:
        return None
    state = {}
    for attr in ['mean_', 'scale_', 'var_', 'n_samples_seen_', 'n_features_in_']:
        if hasattr(scaler, attr):
            val = getattr(scaler, attr)
            if isinstance(val, np.ndarray):
                state[attr] = val.copy()
            elif np.isscalar(val):
                state[attr] = val.item() if hasattr(val, 'item') else val
            else:
                state[attr] = val
    return state


def _extract_pca_state(pca):
    if pca is None:
        return None
    state = {}
    for attr in ['components_', 'mean_', 'explained_variance_', 'explained_variance_ratio_', 'n_components_']:
        if hasattr(pca, attr):
            val = getattr(pca, attr)
            state[attr] = val.copy() if isinstance(val, np.ndarray) else val
    return state


def _estimate_model_size_mb(wrapper_model: torch.nn.Module) -> float:
    n_params = sum(p.numel() for p in wrapper_model.parameters())
    return (n_params * 4) / (1024 ** 2)

In [4]:
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Callable
from tqdm import tqdm


@dataclass
class RunConfig:
    run_name: str
    model_builder: Callable[[int], object]
    n_models: int
    k_top: int
    maturities: list
    oos_start: pd.Timestamp
    gap: int = 0
    refit_freq: int = 1
    benchmark: str = 'hist_mean'
    rsz_maxlags: int = 12
    progress: bool = False
    save_checkpoints: bool = True
    artifacts_root: Path = Path('../artifacts/orchestrator_runs')


def _save_checkpoint(wrapper, seed: int, t_index: int, date_value, run_dir: Path) -> Path:
    ckpt_dir = run_dir / 'checkpoints' / f'seed_{seed:03d}'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / f'step_{t_index:04d}_{pd.Timestamp(date_value).date()}.pt'

    x_scalers_macro_state = None
    if hasattr(wrapper, 'x_scalers_macro') and isinstance(wrapper.x_scalers_macro, dict):
        x_scalers_macro_state = {k: _extract_scaler_state(v) for k, v in wrapper.x_scalers_macro.items()}

    checkpoint = {
        'wrapper_class': wrapper.__class__.__name__,
        'wrapper_module': wrapper.__class__.__module__,
        'torch_state_dict': wrapper.model.state_dict() if hasattr(wrapper, 'model') and wrapper.model is not None else None,
        'best_params_': getattr(wrapper, 'best_params_', None),
        'fit_calls': getattr(wrapper, '_fit_calls', None),
        'x_scaler': _extract_scaler_state(getattr(wrapper, 'x_scaler', None)),
        'x_scaler_forward': _extract_scaler_state(getattr(wrapper, 'x_scaler_forward', None)),
        'x_scaler_fred': _extract_scaler_state(getattr(wrapper, 'x_scaler_fred', None)),
        'x_scalers_macro': x_scalers_macro_state,
        'y_scaler': _extract_scaler_state(getattr(wrapper, 'y_scaler', None)),
        'pca': _extract_pca_state(getattr(wrapper, 'pca', None)),
    }

    torch.save(checkpoint, ckpt_path)
    return ckpt_path


def run_experiment(cfg: RunConfig, X: pd.DataFrame, y_all: np.ndarray, dates: pd.DatetimeIndex):
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_dir = (cfg.artifacts_root / cfg.run_name / ts).resolve()
    run_dir.mkdir(parents=True, exist_ok=True)

    T = len(dates)
    n_outputs = y_all.shape[1] if y_all.ndim > 1 else 1

    all_forecasts = []
    all_val_losses = []
    ckpt_manifest = []

    model_iter = range(cfg.n_models)
    if cfg.progress:
        model_iter = tqdm(model_iter, desc='Seeds')

    for seed in model_iter:
        model = cfg.model_builder(seed)
        val_losses_for_seed = np.full((T, n_outputs), np.nan)

        # This callback is triggered at each refit step by expanding_window.
        def save_cb(model, refit_i, t_index, date_value, **kwargs):
            if hasattr(model, 'val_loss_') and model.val_loss_ is not None:
                val_losses_for_seed[t_index] = model.val_loss_
            if cfg.save_checkpoints:
                ckpt_path = _save_checkpoint(model, seed, t_index, date_value, run_dir)
                ckpt_manifest.append({
                    'seed': seed,
                    'refit_i': refit_i,
                    't_index': int(t_index),
                    'date': str(pd.Timestamp(date_value).date()),
                    'checkpoint_path': str(ckpt_path),
                })

        y_forecast = wu.expanding_window(
            model, X, y_all, dates, cfg.oos_start,
            gap=cfg.gap,
            refit_freq=cfg.refit_freq,
            save_callback=save_cb,
            progress=False,
        )

        all_forecasts.append(y_forecast)
        all_val_losses.append(val_losses_for_seed)

    forecasts_arr = np.stack(all_forecasts, axis=1)
    losses_arr = np.stack(all_val_losses, axis=1)

    ensemble_forecast, topk_indices = compute_top_k_ensemble(forecasts_arr, losses_arr, cfg.k_top)

    r2s = wu.oos_r2(y_all, ensemble_forecast, benchmark=cfg.benchmark)
    pvals = np.array([bu.RSZ_Signif(y_all[:, i], ensemble_forecast[:, i])
                     for i in range(n_outputs)])

    performance_tuples = list(zip(cfg.maturities, r2s.tolist(), pvals.tolist()))

    # Persist arrays and metadata
    np.save(run_dir / 'forecasts_arr.npy', forecasts_arr)
    np.save(run_dir / 'losses_arr.npy', losses_arr)
    np.save(run_dir / 'ensemble_forecast.npy', ensemble_forecast)
    np.save(run_dir / 'topk_indices.npy', topk_indices)
    if cfg.save_checkpoints:
        pd.DataFrame(ckpt_manifest).to_csv(run_dir / 'checkpoint_manifest.csv', index=False)
    
    perf_df = pd.DataFrame(performance_tuples, columns=['maturity', 'r2_oos', 'rsz_pval'])
    perf_df.to_csv(run_dir / 'performance.csv', index=False)

    serializable_cfg = asdict(cfg)
    serializable_cfg['model_builder'] = str(cfg.model_builder)
    serializable_cfg['artifacts_root'] = str(cfg.artifacts_root)
    pd.Series(serializable_cfg).to_json(run_dir / 'run_config.json', indent=2)

    # Storage summary
    if cfg.save_checkpoints:
        ckpt_paths = list((run_dir / 'checkpoints').rglob('*.pt'))
        total_ckpt_bytes = sum(p.stat().st_size for p in ckpt_paths)
        num_checkpoints = len(ckpt_paths)
        total_checkpoint_gb = total_ckpt_bytes / (1024 ** 3)
    else:
        num_checkpoints = 0
        total_checkpoint_gb = 0.0

    summary = {
        'run_dir': str(run_dir),
        'num_checkpoints': num_checkpoints,
        'total_checkpoint_gb': total_checkpoint_gb,
        'save_checkpoints': cfg.save_checkpoints,
        'performance': performance_tuples,
        'forecasts_arr_shape': forecasts_arr.shape,
        'losses_arr_shape': losses_arr.shape,
    }

    return summary

# Backup before chat

In [ ]:
import csv
from models.ann_vector_validation import PyTorchMLPWrapper
from models.macro_forward_ann import MacroForwardANNWrapper
from models.group_ensemble_ann import GroupEnsembleANNWrapper

# Define experiment artifacts root with correct metadata in path
experiment_artifacts_root = Path('/home/ulrikts/Documents/NTNU/TIO4900-Replication/artifacts/orchestrator_runs/run_1971_2018_lw_ANNUAL_NONOVERLAPPING')
experiment_artifacts_root.mkdir(parents=True, exist_ok=True)

# fwd_architectures = [
#     (3,), (5,), (7,),
#     (3, 3), (5, 5), (7, 7),
#     (3, 3, 3), (5, 5, 5), (7, 7, 7),
#     (5, 4, 3),
# ]

macro_architectures = [
    (32,),
    (32, 16),
    # (32, 16, 8),
]

group_ensemble_specs = [
    # 1-layer group ensemble, forward rates direct (pass forward rates with identity mapping)
    {'arch_macro': (1,), 'arch_forward': ()},
    1-layer group ensemble, forward tower 1 layer (3 nodes)
    {'arch_macro': (1,), 'arch_forward': (3,)},
    2-layer group ensemble (2,1), forward tower 2 layers (3,3)
    {'arch_macro': (2, 1), 'arch_forward': (3, 3)},
    3-layer group ensemble (3,2,1), forward tower 3 layers (3,3,3)
    {'arch_macro': (3, 2, 1), 'arch_forward': (3, 3, 3)},
]

def arch_to_name(arch):
    return 'direct' if len(arch) == 0 else '&'.join(str(x) for x in arch)

def base_run_config(run_name, model_builder, n_models=100, k_top=10, save_checkpoints=True):
    return RunConfig(
        run_name=run_name,
        model_builder=model_builder,
        n_models=n_models,
        k_top=k_top,
        maturities=MATURITIES,
        oos_start=pd.Timestamp('1990-01-31'),
        gap=11,
        refit_freq=1,
        benchmark='hist_mean',
        progress=True,
        save_checkpoints=save_checkpoints,
        artifacts_root=experiment_artifacts_root,
    )

def make_fwd_ann_cfg(arch, n_models=100, k_top=10, save_checkpoints=True):
    return base_run_config(
        run_name=f"fwd_ann_{arch_to_name(arch)}_{n_models}runs_top{k_top}",
        model_builder=lambda seed, arch=arch: PyTorchMLPWrapper(
            archi=arch,
            lr=0.01,
            epochs=1000,
            tune_every=60,
            patience=50,
            param_grid={'penalty': [0.01, 0.001, 0.0001]},
            seed=seed,
            use_pca=False,
            n_components=None,
            y_center=True,
        ),
        n_models=n_models,
        k_top=k_top,
        save_checkpoints=save_checkpoints,
    )

def make_macro_forward_cfg(arch_macro, arch_forward=(3,), n_models=100, k_top=10, save_checkpoints=True):
    return base_run_config(
        run_name=(
            f"macro_fwd_ann_fwd{arch_to_name(arch_forward)}_"
            f"macro{arch_to_name(arch_macro)}_{n_models}runs_top{k_top}"
        ),
        model_builder=lambda seed, arch_macro=arch_macro, arch_forward=arch_forward: MacroForwardANNWrapper(
            archi_forward=arch_forward,
            archi_macro=arch_macro,
            lr=0.01,
            epochs=1000,
            tune_every=60,
            patience=50,
            param_grid={'penalty': [0.001, 0.0001], 'dropout_rate': [0.0, 0.1, 0.3]},
            seed=seed,
            y_center=True,
        ),
        n_models=n_models,
        k_top=k_top,
        save_checkpoints=save_checkpoints,
    )

def make_group_ensemble_cfg(arch_macro, arch_forward, n_models=100, k_top=10, save_checkpoints=True):
    return base_run_config(
        run_name=(
            f"group_ens_ann_fwd{arch_to_name(arch_forward)}_"
            f"grp{arch_to_name(arch_macro)}_{n_models}runs_top{k_top}"
        ),
        model_builder=lambda seed, arch_macro=arch_macro, arch_forward=arch_forward: GroupEnsembleANNWrapper(
            archi_forward=arch_forward,
            archi_macro=arch_macro,
            lr=0.01,
            epochs=1000,
            tune_every=60,
            patience=50,
            param_grid={'penalty': [0.001, 0.0001], 'dropout_rate': [0.0, 0.1, 0.3]},
            seed=seed,
            y_center=True,
        ),
        n_models=n_models,
        k_top=k_top,
        save_checkpoints=save_checkpoints,
    )

run_configs = []

# Forward-only ANN grid (restricted list above).
# run_configs += [make_fwd_ann_cfg(arch, save_checkpoints=False) for arch in fwd_architectures]

# Macro + forward two-tower ANN grid from init-style defaults.
# run_configs += [make_macro_forward_cfg(arch_macro, save_checkpoints=False, n_models=100) for arch_macro in macro_architectures]

# Group-ensemble ANN grid from image specs + init-style defaults.
# run_configs += [
#     make_group_ensemble_cfg(spec['arch_macro'], spec['arch_forward'], save_checkpoints=False, n_models=100)
#     for spec in group_ensemble_specs
# ]

run_configs = [make_fwd_ann_cfg((3,), save_checkpoints=False, n_models=100), make_macro_forward_cfg((32,), (3,), save_checkpoints=False, n_models=100)]

master_summary_path = (experiment_artifacts_root / 'master_summary.csv').resolve()
master_summary_path.parent.mkdir(parents=True, exist_ok=True)

master_fields = [
    'timestamp', 'run_name', 'status', 'run_dir', 'num_checkpoints',
    'total_checkpoint_gb', 'forecasts_arr_shape', 'losses_arr_shape',
    'performance', 'error',
]

def append_master_summary(row_dict):
    write_header = not master_summary_path.exists()
    with open(master_summary_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=master_fields)
        if write_header:
            writer.writeheader()
        writer.writerow(row_dict)

all_summaries = []
failed_runs = []

for i, cfg in enumerate(run_configs, start=1):
    print(f"[{i}/{len(run_configs)}] Running {cfg.run_name} ...")
    ts_now = datetime.now().isoformat(timespec='seconds')

    try: 
        summary = run_experiment(cfg, X, y_all, dates) # X vs X_realtime is the choice of using lagged vs real-time macro data.
        all_summaries.append(summary)

        append_master_summary({
            'timestamp': ts_now,
            'run_name': cfg.run_name,
            'status': 'ok',
            'run_dir': summary.get('run_dir', ''),
            'num_checkpoints': summary.get('num_checkpoints', ''),
            'total_checkpoint_gb': summary.get('total_checkpoint_gb', ''),
            'forecasts_arr_shape': summary.get('forecasts_arr_shape', ''),
            'losses_arr_shape': summary.get('losses_arr_shape', ''),
            'performance': summary.get('performance', ''),
            'error': '',
        })

    except Exception as e:
        err = f"{type(e).__name__}: {e}"
        failed_runs.append({'run_name': cfg.run_name, 'error': err})
        print(f"FAILED {cfg.run_name} -> {err}")

        append_master_summary({
            'timestamp': ts_now,
            'run_name': cfg.run_name,
            'status': 'failed',
            'run_dir': '',
            'num_checkpoints': '',
            'total_checkpoint_gb': '',
            'forecasts_arr_shape': '',
            'losses_arr_shape': '',
            'performance': '',
            'error': err,
        })

print(f"Master summary: {master_summary_path}")

print(f"Completed {len(all_summaries)} successful runs out of {len(run_configs)} total.")

print(f"Failed runs: {len(failed_runs)}")

# Main block for running GAP=11 MODELS:

In [ ]:
import csv
from models.ann_vector_validation import PyTorchMLPWrapper
from models.macro_forward_ann import MacroForwardANNWrapper
from models.group_ensemble_ann import GroupEnsembleANNWrapper

# Define experiment artifacts root with correct metadata in path
experiment_artifacts_root = Path('/home/ulrikts/Documents/NTNU/TIO4900-Replication/artifacts/orchestrator_runs/run_1971_2018_lw_ANNUAL_NONOVERLAPPING')
experiment_artifacts_root.mkdir(parents=True, exist_ok=True)

# Model families shown in the screenshot, expressed as declarative config specs.
macro_forward_specs = [
    {
        'label': 'NN 1layer (32 nodes), fwd rates direct',
        'arch_macro': (32,),
        'arch_forward': (),
    },
    {
        'label': 'NN 1layer (32, 16 nodes), fwd rates direct',
        'arch_macro': (32, 16),
        'arch_forward': (),
    },
    {
        'label': 'NN 1layer (32, 16, 8 nodes), fwd rates direct',
        'arch_macro': (32, 16, 8),
        'arch_forward': (),
    },
    {
        'label': 'NN 1layer (32 nodes), fwd rates net',
        'arch_macro': (32,),
        'arch_forward': (3,),
    },
    {
        'label': 'NN 1layer (32, 16 nodes), fwd rates net',
        'arch_macro': (32, 16),
        'arch_forward': (3,),
    },
    {
        'label': 'NN 1layer (32, 16, 8 nodes), fwd rates net',
        'arch_macro': (32, 16, 8),
        'arch_forward': (3,),
    },
]

group_ensemble_specs = [
    {
        'label': 'NN 1layer group ensemble (1 node per group), fwd rates direct',
        'arch_macro': (1,),
        'arch_forward': (),
    },
    {
        'label': 'NN 2layer group ensemble (1 node per group), fwd rates direct',
        'arch_macro': (2, 1),
        'arch_forward': (),
    },
    {
        'label': 'NN 3layer group ensemble (1 node per group), fwd rates direct',
        'arch_macro': (3, 2, 1),
        'arch_forward': (),
    },
    {
        'label': 'NN 1layer group ensemble (1 node per group), fwd rates net',
        'arch_macro': (1,),
        'arch_forward': (3,),
    },
    {
        'label': 'NN 2layer group ensemble (1 node per group), fwd rates net',
        'arch_macro': (2, 1),
        'arch_forward': (3, 3),
    },
    {
        'label': 'NN 3layer group ensemble (1 node per group), fwd rates net',
        'arch_macro': (3, 2, 1),
        'arch_forward': (3, 3, 3),
    },
]

def arch_to_name(arch):
    return 'direct' if len(arch) == 0 else '&'.join(str(x) for x in arch)


def base_run_config(run_name, model_builder, n_models=100, k_top=10, save_checkpoints=True):
    return RunConfig(
        run_name=run_name,
        model_builder=model_builder,
        n_models=n_models,
        k_top=k_top,
        maturities=MATURITIES,
        oos_start=pd.Timestamp('1990-01-31'),
        gap=11,
        refit_freq=1,
        benchmark='hist_mean',
        progress=True,
        save_checkpoints=save_checkpoints,
        artifacts_root=experiment_artifacts_root,
    )


def make_fwd_ann_cfg(arch, n_models=100, k_top=10, save_checkpoints=True):
    return base_run_config(
        run_name=f"fwd_ann_{arch_to_name(arch)}_{n_models}runs_top{k_top}",
        model_builder=lambda seed, arch=arch: PyTorchMLPWrapper(
            archi=arch,
            lr=0.01,
            epochs=1000,
            tune_every=60,
            patience=50,
            param_grid={'penalty': [0.01, 0.001, 0.0001]},
            seed=seed,
            use_pca=False,
            n_components=None,
            y_center=True,
        ),
        n_models=n_models,
        k_top=k_top,
        save_checkpoints=save_checkpoints,
    )


def make_macro_forward_cfg(arch_macro, arch_forward=(3,), n_models=100, k_top=10, save_checkpoints=True):
    return base_run_config(
        run_name=(
            f"macro_fwd_ann_fwd{arch_to_name(arch_forward)}_"
            f"macro{arch_to_name(arch_macro)}_{n_models}runs_top{k_top}"
        ),
        model_builder=lambda seed, arch_macro=arch_macro, arch_forward=arch_forward: MacroForwardANNWrapper(
            archi_forward=arch_forward,
            archi_macro=arch_macro,
            lr=0.01,
            epochs=1000,
            tune_every=60,
            patience=50,
            param_grid={'penalty': [0.001, 0.0001], 'dropout_rate': [0.0, 0.1, 0.3]},
            seed=seed,
            y_center=True,
        ),
        n_models=n_models,
        k_top=k_top,
        save_checkpoints=save_checkpoints,
    )


def make_group_ensemble_cfg(arch_macro, arch_forward, n_models=100, k_top=10, save_checkpoints=True):
    return base_run_config(
        run_name=(
            f"group_ens_ann_fwd{arch_to_name(arch_forward)}_"
            f"grp{arch_to_name(arch_macro)}_{n_models}runs_top{k_top}"
        ),
        model_builder=lambda seed, arch_macro=arch_macro, arch_forward=arch_forward: GroupEnsembleANNWrapper(
            archi_forward=arch_forward,
            archi_macro=arch_macro,
            lr=0.01,
            epochs=1000,
            tune_every=60,
            patience=50,
            param_grid={'penalty': [0.001, 0.0001], 'dropout_rate': [0.0, 0.1, 0.3]},
            seed=seed,
            y_center=True,
        ),
        n_models=n_models,
        k_top=k_top,
        save_checkpoints=save_checkpoints,
    )


run_configs = [
    *(make_macro_forward_cfg(spec['arch_macro'], spec['arch_forward'], save_checkpoints=False, n_models=100)
      for spec in macro_forward_specs),
    *(make_group_ensemble_cfg(spec['arch_macro'], spec['arch_forward'], save_checkpoints=False, n_models=100)
      for spec in group_ensemble_specs),
]

master_summary_path = (experiment_artifacts_root / 'master_summary.csv').resolve()
master_summary_path.parent.mkdir(parents=True, exist_ok=True)

master_fields = [
    'timestamp', 'run_name', 'status', 'run_dir', 'num_checkpoints',
    'total_checkpoint_gb', 'forecasts_arr_shape', 'losses_arr_shape',
    'performance', 'error',
]

def append_master_summary(row_dict):
    write_header = not master_summary_path.exists()
    with open(master_summary_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=master_fields)
        if write_header:
            writer.writeheader()
        writer.writerow(row_dict)

all_summaries = []
failed_runs = []

for i, cfg in enumerate(run_configs, start=1):
    print(f"[{i}/{len(run_configs)}] Running {cfg.run_name} ...")
    ts_now = datetime.now().isoformat(timespec='seconds')

    try:
        summary = run_experiment(cfg, X, y_all, dates)  # X vs X_realtime is the choice of using lagged vs real-time macro data.
        all_summaries.append(summary)

        append_master_summary({
            'timestamp': ts_now,
            'run_name': cfg.run_name,
            'status': 'ok',
            'run_dir': summary.get('run_dir', ''),
            'num_checkpoints': summary.get('num_checkpoints', ''),
            'total_checkpoint_gb': summary.get('total_checkpoint_gb', ''),
            'forecasts_arr_shape': summary.get('forecasts_arr_shape', ''),
            'losses_arr_shape': summary.get('losses_arr_shape', ''),
            'performance': summary.get('performance', ''),
            'error': '',
        })

    except Exception as e:
        err = f"{type(e).__name__}: {e}"
        failed_runs.append({'run_name': cfg.run_name, 'error': err})
        print(f"FAILED {cfg.run_name} -> {err}")

        append_master_summary({
            'timestamp': ts_now,
            'run_name': cfg.run_name,
            'status': 'failed',
            'run_dir': '',
            'num_checkpoints': '',
            'total_checkpoint_gb': '',
            'forecasts_arr_shape': '',
            'losses_arr_shape': '',
            'performance': '',
            'error': err,
        })

print(f"Master summary: {master_summary_path}")
print(f"Completed {len(all_summaries)} successful runs out of {len(run_configs)} total.")
print(f"Failed runs: {len(failed_runs)}")